# 05 — RAG clásico (baseline denso, lector vía Ollama)

Retrieval denso top-5 con all-MiniLM-L6-v2 + lector vía **ChatOllama** (`READER_OLLAMA_MODEL`: `qwen2.5:1.5b` para paridad con el baseline en CPU, `qwen2.5:7b` para el run con modelo más grande) + mismo scorer. El retrieval corre igual que en local (`rag_baseline/retrieve.py`); el lector es el mismo prompt y el mismo `condense()` de `answer.py`, solo cambia el modelo que contesta (Ollama en vez de transformers en CPU).

**Prerrequisito:** notebook 01 (chunks). En GPU el embedder usa CUDA solo.

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros ---
READER_OLLAMA_MODEL = "qwen2.5:1.5b"   # lector: 1.5b (paridad CPU) o qwen2.5:7b (mas grande)
K = 5
RETRIEVAL_FILE = REPO + "/rag_baseline/retrieval.json"
TRACE_OUT = REPO + "/rag_baseline/run_trace_colab.jsonl"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 sentence-transformers==6.3.0 numpy>=2.5.5 langchain-ollama

## Ollama (setup compacto e idempotente)

Este notebook usa modelos intermedios vía **Ollama** corriendo en la propia instancia de Colab (un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook). Las celdas son idempotentes: verifican si el binario existe, si el servidor ya responde y si el modelo ya está bajado antes de hacer nada. La versión completa, con la explicación y las fuentes, está en `00_ollama_setup.ipynb`.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

In [ ]:
%%bash
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
ollama list

In [ ]:
from langchain_ollama import ChatOllama

_llm = ChatOllama(model=READER_OLLAMA_MODEL, base_url="http://localhost:11434", temperature=0)
print(_llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("ChatOllama OK contra", READER_OLLAMA_MODEL)

## 1. Retrieval denso top-5

Corre `rag_baseline/retrieve.py` tal cual: embebe las 151 páginas con all-MiniLM-L6-v2 y recupera top-5 por pregunta, con el diagnóstico de cobertura de cifras esperadas (separa fallos de retrieval de fallos de lector).

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "retrieve.py", "--k", str(K),
                    "--out", RETRIEVAL_FILE], cwd=REPO + "/rag_baseline")
print("exit:", r.returncode)

## 2. Lector vía Ollama

Mismo sistema y mismo `condense()` que `rag_baseline/answer.py`: el contexto son los 5 chunks recuperados (ventanas de ~1,800 caracteres por chunk, 2 ventanas) y el modelo responde solo desde ellos. La traza tiene el mismo formato que la de CPU, así que el scorer no cambia.

In [ ]:
import json, sys
sys.path.insert(0, REPO + "/rag_baseline")
from answer import SYSTEM, condense
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

reader = ChatOllama(model=READER_OLLAMA_MODEL, base_url="http://localhost:11434",
                    temperature=0, num_predict=100)
print("lector RAG (Ollama):", READER_OLLAMA_MODEL)

data = json.load(open(RETRIEVAL_FILE))
with open(TRACE_OUT, "w") as f:
    for item in data["questions"]:
        condensed = [condense(r["text"], item["question"]) for r in item["retrieved"]]
        ctx = "\n\n".join(f"[Página {r['page']}]\n{c}"
                          for r, c in zip(item["retrieved"], condensed))
        user = (f"Fragmentos del informe anual de FEMSA 2024:\n\n{ctx}\n\n"
                f"Pregunta: {item['question']}")
        ans = reader.invoke([SystemMessage(content=SYSTEM),
                             HumanMessage(content=user)]).content.strip()
        rec = {"id": item["id"], "level": item["level"],
               "question": item["question"], "decomposition": "rag",
               "reader_model": f"ollama {READER_OLLAMA_MODEL}",
               "retrieved_pages": [r["page"] for r in item["retrieved"]],
               "retrieval_scores": [r["score"] for r in item["retrieved"]],
               "context": ctx, "final_answer": ans}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print(f"{item['id']} -> {ans[:160]}", flush=True)
print(f"traza -> {TRACE_OUT}")

## 3. Puntuar con el mismo scorer

Corre `rag_baseline/score.py` tal cual: misma exactitud numérica, misma definición de alucinación, misma regla Q3 con 1,596. Compara con el baseline en CPU (2/12: Q6 y Q11).

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "score.py", "--trace", TRACE_OUT,
                    "--retrieval", RETRIEVAL_FILE,
                    "--out", REPO + "/rag_baseline/results_colab.md"],
                   cwd=REPO + "/rag_baseline")
print("exit:", r.returncode)

## Alternativa documentada (Hugging Face, no por defecto en Colab)

Si quieres el lector HF en GPU en vez de Ollama, corre el script original con el modelo local (hay que bajar Qwen2.5-1.5B-Instruct a `memo/models/` o apuntar `--model` a `Qwen/Qwen2.5-1.5B-Instruct` en el Hub):

```bash
python rag_baseline/answer.py --retrieval rag_baseline/retrieval.json --model Qwen/Qwen2.5-1.5B-Instruct --out rag_baseline/run_trace_hf.jsonl
```

Ojo: `answer.py` carga el modelo sin `device_map`, así que en Colab correría en CPU salvo que se adapte. Por eso el default en Colab es Ollama.